# A2AJ Canadian Hansard — Hugging Face Dataset

> **Alpha — work in progress.** This is an alpha version of a work in progress academic research project. It is likely to undergo significant revisions.

**Documentation:**

- Canadian Hansard Dataset: https://huggingface.co/datasets/a2aj/hansard
- Hugging Face Datasets: https://huggingface.co/docs/datasets/main/en/index

Hugging Face Datasets is a library that provides easy access to datasets stored in a standardized format and ready to use with just a few lines of code. Instead of downloading files manually and dealing with different formats, the Datasets library handles the heavy lifting — loading data efficiently, managing memory, and providing tools for processing.

The A2AJ (Access to Algorithmic Justice) Canadian Hansard dataset provides bulk access to parliamentary Hansard transcripts — the official record of debate — for Canadian legislatures. The text is parsed into structured rows with **one row per intervention** (a speaker turn or a procedural note), alongside metadata such as the sitting date, speaker, order/subject of business, and jurisdiction. It currently covers the Legislative Assembly of Ontario.

The dataset is built for **computational legal and political research**: programmatically searching the parliamentary record for patterns, tracking how debate and policy evolve over time and in light of case law, assembling evidence of legislative purpose and intent, and linking parliamentary discussion to legislation and case law.

This notebook shows how to load a jurisdiction, convert it to a pandas DataFrame, explore the structure, and run a few example research queries.

**Note:** This bulk approach is ideal for analysis, machine learning, or offline work. If you only need a few specific documents, you can also read the Parquet file directly (shown below).


## Setup

In [2]:
# install required packages if not already installed
# !pip install datasets pandas

from datasets import load_dataset
import pandas as pd

/home/sr/a2aj-hansard/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load a specific legislature

Each jurisdiction is a separate **config**, each with a single `train` split.

Current options are:

| Config | Legislature |
|---|---|
| `ontario` | Legislative Assembly of Ontario |

In [3]:
# load one jurisdiction (config), e.g. Ontario
hansard = load_dataset("a2aj/hansard", "ontario", split="train")

# convert to a DataFrame
df = hansard.to_pandas()
df.head(5)

Generating train split: 100%|██████████| 1347812/1347812 [00:01<00:00, 702940.76 examples/s]


,ID,Date,jurisdiction,chamber,language,OrderofBusiness,SubjectofBusiness,PersonSpeaking,intervention_type,Intervention,upstream_license,source_url
0,19740305-0000001,1974-03-05,ontario,legislative_assembly,en,NaN,NaN,NaN,procedural,"Tuesday, March 5, 1974, being the first day of...","See upstream license, including non-commercial...",https://www.ola.org/en/legislative-business/ho...
1,19740305-0000002,1974-03-05,ontario,legislative_assembly,en,NaN,SPEECH FROM THE THRONE,Hon. W. Ross Macdonald (Lieutenant Governor),speech,Pray be seated. Mr. Speaker and members of the...,"See upstream license, including non-commercial...",https://www.ola.org/en/legislative-business/ho...
2,19740305-0000003,1974-03-05,ontario,legislative_assembly,en,NaN,SPEECH FROM THE THRONE,NaN,procedural,Prayers.,"See upstream license, including non-commercial...",https://www.ola.org/en/legislative-business/ho...
3,19740305-0000004,1974-03-05,ontario,legislative_assembly,en,NaN,SPEECH FROM THE THRONE,Mr. Speaker,speech,"To prevent mistakes, I have obtained a copy of...","See upstream license, including non-commercial...",https://www.ola.org/en/legislative-business/ho...
4,19740305-0000005,1974-03-05,ontario,legislative_assembly,en,NaN,SPEECH FROM THE THRONE,Mr. R. F. Nixon (Leader of the Opposition),speech,"Mr. Speaker, on a point of order, if you will ...","See upstream license, including non-commercial...",https://www.ola.org/en/legislative-business/ho...


## Alternative: read the Parquet files directly

The dataset is also offered in Parquet, with one file per jurisdiction. You can read a file directly with pandas (no `datasets` dependency) — handy for quick local use.

In [4]:
import pandas as pd

df = pd.read_parquet("hf://datasets/a2aj/hansard/ontario/train.parquet")
df.head(5)

,ID,Date,jurisdiction,chamber,language,OrderofBusiness,SubjectofBusiness,PersonSpeaking,intervention_type,Intervention,upstream_license,source_url
0,19740305-0000001,1974-03-05,ontario,legislative_assembly,en,NaN,NaN,NaN,procedural,"Tuesday, March 5, 1974, being the first day of...","See upstream license, including non-commercial...",https://www.ola.org/en/legislative-business/ho...
1,19740305-0000002,1974-03-05,ontario,legislative_assembly,en,NaN,SPEECH FROM THE THRONE,Hon. W. Ross Macdonald (Lieutenant Governor),speech,Pray be seated. Mr. Speaker and members of the...,"See upstream license, including non-commercial...",https://www.ola.org/en/legislative-business/ho...
2,19740305-0000003,1974-03-05,ontario,legislative_assembly,en,NaN,SPEECH FROM THE THRONE,NaN,procedural,Prayers.,"See upstream license, including non-commercial...",https://www.ola.org/en/legislative-business/ho...
3,19740305-0000004,1974-03-05,ontario,legislative_assembly,en,NaN,SPEECH FROM THE THRONE,Mr. Speaker,speech,"To prevent mistakes, I have obtained a copy of...","See upstream license, including non-commercial...",https://www.ola.org/en/legislative-business/ho...
4,19740305-0000005,1974-03-05,ontario,legislative_assembly,en,NaN,SPEECH FROM THE THRONE,Mr. R. F. Nixon (Leader of the Opposition),speech,"Mr. Speaker, on a point of order, if you will ...","See upstream license, including non-commercial...",https://www.ola.org/en/legislative-business/ho...


## Explore the data structure

Each row is one intervention. The fields are:

| Field | Description |
|---|---|
| `ID` | Stable per-row identifier (sitting date + sequence number). |
| `Date` | Sitting date (YYYY-MM-DD). |
| `jurisdiction` | Which legislature (e.g. `ontario`). |
| `chamber` | Chamber within the legislature (e.g. `legislative_assembly`). |
| `language` | Language of the source page (e.g. `en`). |
| `OrderofBusiness` | High-level section heading (the order of business). |
| `SubjectofBusiness` | Specific topic / matter within the order of business. |
| `PersonSpeaking` | Speaker label, or empty for procedural rows. |
| `intervention_type` | `speech` (a speaker turn) or `procedural` (a procedural note). |
| `Intervention` | The text of the intervention. |
| `upstream_license` | License / attribution terms from the source legislature. |
| `source_url` | The day-page URL this row was parsed from. |

In [5]:
print("rows:", len(df))
print("columns:", list(df.columns))
print("date range:", df["Date"].min(), "->", df["Date"].max())

# speech vs procedural
df["intervention_type"].value_counts()

rows: 1347812
columns: ['ID', 'Date', 'jurisdiction', 'chamber', 'language', 'OrderofBusiness', 'SubjectofBusiness', 'PersonSpeaking', 'intervention_type', 'Intervention', 'upstream_license', 'source_url']
date range: 1974-03-05 -> 2026-06-02


intervention_type
speech        1275424
procedural      72388
Name: count, dtype: int64

In [6]:
# most frequent speakers
df["PersonSpeaking"].value_counts().head(10)

PersonSpeaking
The Speaker (Hon. Dave Levac)          38622
Mr. Speaker                            37879
The Speaker (Hon. Ted Arnott)          32837
The Speaker                            27464
The Acting Speaker                     17016
The Speaker (Hon. Steve Peters)        16581
The Deputy Speaker                     14008
Hon. Mr. Davis                         12193
Mr. Chairman                           10000
The Acting Speaker (Mr. Ted Arnott)     9096
Name: count, dtype: int64

## Example research queries

A few illustrative patterns. Because every intervention is a row with a speaker, date, topic, and full text, you can slice the parliamentary record however your research question requires.

In [7]:
# 1) Keyword search across the full text of interventions
term = "climate change"
hits = df[df["Intervention"].str.contains(term, case=False, na=False)]
print(f"{len(hits):,} interventions mention '{term}'")
hits[["Date", "PersonSpeaking", "OrderofBusiness", "Intervention"]].head(5)

3,864 interventions mention 'climate change'


,Date,PersonSpeaking,OrderofBusiness,Intervention
312060,1982-10-08,Mr. Riddell,ORDERS OF THE DAY,This in turn requires as little uncertainty as...
459794,1988-12-14,Ms. Collins,ORAL QUESTIONS,My question is for the Minister of Government ...
460785,1989-01-05,Mr. McGuigan,PRIVATE MEMBERS’ PUBLIC BUSINESS,The Encyclopaedia Britannica defines “land ref...
494126,1990-03-28,Mr Daigeler,REPORTS BY COMMITTEES,Mrs Sullivan from the select committee on ener...
507129,1990-06-28,Hon Mr Ward,ORDERS OF THE DAY,Mr Ward moved resolution 37: That the followin...


In [8]:
# 2) Filter to a date range (e.g. 2024 onward)
dates = pd.to_datetime(df["Date"], errors="coerce")
recent = df[dates >= "2024-01-01"]
print(f"{len(recent):,} interventions since 2024-01-01")
recent.head(5)

53,642 interventions since 2024-01-01


,ID,Date,jurisdiction,chamber,language,OrderofBusiness,SubjectofBusiness,PersonSpeaking,intervention_type,Intervention,upstream_license,source_url
1294170,20240220-0000001,2024-02-20,ontario,legislative_assembly,en,NaN,NaN,NaN,procedural,The House met at 1015.,"See upstream license, including non-commercial...",https://www.ola.org/en/legislative-business/ho...
1294171,20240220-0000002,2024-02-20,ontario,legislative_assembly,en,NaN,NaN,The Speaker (Hon. Ted Arnott),speech,Good morning. Let us pray.,"See upstream license, including non-commercial...",https://www.ola.org/en/legislative-business/ho...
1294172,20240220-0000003,2024-02-20,ontario,legislative_assembly,en,NaN,NaN,NaN,procedural,Prayers.,"See upstream license, including non-commercial...",https://www.ola.org/en/legislative-business/ho...
1294173,20240220-0000004,2024-02-20,ontario,legislative_assembly,en,NaN,Resignation of member for Milton,The Speaker (Hon. Ted Arnott),speech,"I beg to inform the House that, during the adj...","See upstream license, including non-commercial...",https://www.ola.org/en/legislative-business/ho...
1294174,20240220-0000005,2024-02-20,ontario,legislative_assembly,en,NaN,Tabling of sessional papers,The Speaker (Hon. Ted Arnott),speech,"I beg to inform the House that, during the adj...","See upstream license, including non-commercial...",https://www.ola.org/en/legislative-business/ho...


In [9]:
# 3) How a topic is discussed over time (mentions per year)
term = "housing"
mask = df["Intervention"].str.contains(term, case=False, na=False)
years = pd.to_datetime(df.loc[mask, "Date"], errors="coerce").dt.year
years.value_counts().sort_index()

Date
1974     927
1975     756
1976     512
1977     274
1978     554
1979     289
1980     309
1981     534
1982     528
1983     405
1984     296
1985     285
1986     574
1987     733
1988     834
1989     961
1990     473
1991     625
1992     476
1993     382
1994     495
1995     221
1996     571
1997     810
1998     269
1999     205
2000     477
2001     251
2002     281
2003      90
2004     378
2005     421
2006     442
2007     273
2008     469
2009     489
2010     356
2011     238
2012     353
2013     561
2014     221
2015     458
2016     585
2017     685
2018     784
2019     587
2020     639
2021     754
2022    1413
2023    1907
2024    1427
2025     995
2026     732
Name: count, dtype: int64

In [10]:
# 4) Everything a particular speaker said on a given day
day = df[(df["Date"] == "2024-03-21") & (df["intervention_type"] == "speech")]
day[["PersonSpeaking", "SubjectofBusiness", "Intervention"]].head(10)

,PersonSpeaking,SubjectofBusiness,Intervention
1299217,The Speaker (Hon. Ted Arnott),NaN,Good morning. Let us pray. Prières / Prayers.
1299218,The Speaker (Hon. Ted Arnott),"Croatian Heritage Day Act, 2024 / Loi de 2024 ...",Ms. Skelly moved third reading of the followin...
1299219,The Speaker (Hon. Ted Arnott),"Croatian Heritage Day Act, 2024 / Loi de 2024 ...",I recognize the member for Flamborough–Glanbrook.
1299220,Ms. Donna Skelly,"Croatian Heritage Day Act, 2024 / Loi de 2024 ...","Good morning, everyone. I am so pleased to ris..."
1299221,The Acting Speaker (Ms. Patrice Barnes),"Croatian Heritage Day Act, 2024 / Loi de 2024 ...",Further debate?
1299222,Mrs. Daisy Wai,"Croatian Heritage Day Act, 2024 / Loi de 2024 ...","Today, we gather in the Legislature to celebra..."
1299223,The Acting Speaker (Ms. Patrice Barnes),"Croatian Heritage Day Act, 2024 / Loi de 2024 ...",Further debate?
1299224,MPP Jamie West,"Croatian Heritage Day Act, 2024 / Loi de 2024 ...",I’ll be sharing my time with the member from N...
1299225,The Acting Speaker (Ms. Patrice Barnes),"Croatian Heritage Day Act, 2024 / Loi de 2024 ...",Further debate?
1299226,M me France Gélinas,"Croatian Heritage Day Act, 2024 / Loi de 2024 ...",I do agree that this is a beautiful letter tha...


## Licensing

The code used to create the dataset is under an open-source MIT license. Users must also comply with the upstream licenses recorded in the `upstream_license` field, which may include limits on commercial use and other restrictions. The texts here are unofficial copies — for authoritative versions, consult the `source_url`. See the [dataset card](https://huggingface.co/datasets/a2aj/hansard) for full details.

Contact: a2aj@yorku.ca